# 双引擎优化器：AdamW + Muon

源码导航：[`core/utils/walkie_optim.py`](../../../core/utils/walkie_optim.py) 中的 `zeropower_via_newton_schulz5`、`Muon`、`split_walkie_params`、`build_walkie_optimizers`。

## 1. 理论背景

### 1.1 AdamW 在高维矩阵权重上的局限

标准 AdamW 为每个参数独立维护一阶矩 $m_t$ 和二阶矩 $v_t$，参数更新量为：

$$\Delta \theta_t = -\alpha \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

对标量/1D 参数，这种**逐元素自适应**非常合适。但对于高维矩阵权重（如 $W_Q \in \mathbb{R}^{d \times d_k}$），梯度矩阵的不同奇异方向对应的步长高度不均衡：某些方向被缩放得过大（梯度方差小），另一些方向几乎不动（梯度方差大），导致参数空间利用率低。

### 1.2 Muon：动量 + Newton-Schulz 正交化

**Muon（Momentum Orthogonal Update）** 由 Keller Jordan 提出。其核心思想是：对动量更新 $U_t$（shape 同参数矩阵），不直接用 $U_t$ 做参数更新，而是先求其**极分解**的正交因子 $Q_t$，使每个奇异方向的步长等效归一：

$$G_t = \text{gradient}, \quad U_t = \beta U_{t-1} + G_t \quad \text{(Nesterov)}$$
$$Q_t = \text{NS5}(U_t) \approx \text{polar orthogonal factor of } U_t$$
$$\theta_{t+1} = \theta_t - \alpha \cdot s \cdot Q_t - \alpha \lambda \theta_t$$

其中 $s = \max(1, \sqrt{\text{fan\_out}/\text{fan\_in}})$ 是对矩阵宽高比的尺度补偿，$\lambda$ 为权重衰减系数。

### 1.3 Newton-Schulz 5 阶迭代

设 $G$ 的奇异值分解为 $G = U \Sigma V^T$，极分解的正交因子为 $\hat{G} = UV^T$（即令所有奇异值为 1）。Newton-Schulz 迭代通过如下 5 阶多项式递推逼近 $\hat{G}$：

$$X_{k+1} = a X_k + b X_k X_k^T X_k + c X_k (X_k^T X_k)^2, \quad (a,b,c) = (3.4445,\ -4.7750,\ 2.0315)$$

初始化 $X_0 = G / \|G\|_F$（Frobenius 范数归一）。迭代 5 次后 $X_5 \approx UV^T$，计算量 $O(\min(m,n)^2 \max(m,n))$，无需 SVD。

实现上做了两个优化：
- **短边在前**：若 $m > n$，令 $X_0 = G^T / \|G\|$ 在 $n \times m$ 形状上迭代，最终转置回来，减小中间张量 $XX^T$ 的大小
- **bf16 加速**：在 CUDA 上以 `bfloat16` 执行迭代，结果 cast 回原始精度

### 1.4 参数分组规则

| 参数类型 | 优化器 | 原因 |
|---|---|---|
| 2D 矩阵权重（Q/K/V/O、gate/up/down FFN） | **Muon** | 极分解使各奇异方向步长均衡 |
| Embedding（`tok_embeddings`）| AdamW | 行稀疏更新，逐元素自适应更合适 |
| LM head（`lm_head`，tied） | AdamW | 与 embedding 共享参数，只需加入一次 |
| RMSNorm scale | AdamW | 1D 向量，无矩阵奇异方向问题 |
| Bias | AdamW | 1D 向量 |

### 1.5 Tied Weights 的重复梯度问题

Walkie 中 `lm_head.weight` 与 `tok_embeddings.weight` 是同一个 `nn.Parameter` 对象（`id` 相同）。如果不加处理，`model.named_parameters()` 会以两个不同名称各返回它一次，导致同一个参数同时进入两个参数组，反向传播后该参数会被**更新两次**（相当于 lr 翻倍），破坏训练稳定性。

`split_walkie_params` 通过维护 `seen: set[int]` 集合记录已分配的参数 `id`：

```python
seen: set[int] = set()
for name, p in model.named_parameters():
    if id(p) in seen:   # 已经分配过（tied weight 的另一个名称），跳过
        continue
    seen.add(id(p))
    ...                 # 正常分组
```

这样无论 `lm_head.weight` 还是 `tok_embeddings.weight` 先出现，另一个都会被跳过，整个参数只被加入 AdamW 组一次。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.utils.walkie_optim import (
    Muon,
    build_walkie_optimizers,
    split_walkie_params,
    zeropower_via_newton_schulz5,
)
from core.model.walkie import WalkieConfig, WalkieForCausalLM

### 3. Newton-Schulz 正交化验证

In [ ]:
torch.manual_seed(0)

# 模拟一个 Q 投影矩阵的梯度：(out_features, in_features) = (256, 128)
G = torch.randn(256, 128)

U = zeropower_via_newton_schulz5(G, steps=5)

print(f"G shape: {tuple(G.shape)}")
print(f"U shape: {tuple(U.shape)}")

# 验证 U 的行向量近似正交：U @ U^T 应接近单位矩阵
eye_approx = U @ U.T
off_diag_err = (eye_approx - torch.eye(256)).abs().max().item()
print(f"U @ U^T 与 I 的最大误差: {off_diag_err:.4f}  (应 < 0.1)")

# 验证每行范数 ≈ 1（正交矩阵的行/列均为单位向量）
row_norms = U.norm(dim=1)
print(f"行范数均值: {row_norms.mean().item():.4f}  |  行范数标准差: {row_norms.std().item():.4f}")

### 4. 参数分组验证

In [ ]:
cfg = WalkieConfig(
    vocab_size=128, block_size=32, n_embd=64,
    n_layer=2, n_head=4, n_head_kv=2, head_dim=16, d_ffn=128,
    bias=False, tie_weights=True,
)
model = WalkieForCausalLM(cfg)

muon_params, adamw_params, muon_names, adamw_names = split_walkie_params(model)

print(f"Muon  params: {len(muon_params)} 组，总参数量 {sum(p.numel() for p in muon_params):,}")
print(f"AdamW params: {len(adamw_params)} 组，总参数量 {sum(p.numel() for p in adamw_params):,}")

print("\nMuon 管理的参数（全部应为 2D 矩阵权重，无 embed/norm/lm_head）:")
for name, p in zip(muon_names, muon_params):
    print(f"  {name:55s}  shape={tuple(p.shape)}")

print("\nAdamW 管理的参数（norm scale、embedding 等）:")
for name, p in zip(adamw_names, adamw_params):
    print(f"  {name:55s}  shape={tuple(p.shape)}")

# ===== Tied Weights 验证 =====
# lm_head.weight 与 tok_embeddings.weight 同为一个 Parameter 对象
# split_walkie_params 通过 seen=set(id(p)) 确保它只进入 AdamW 一次
all_param_ids_in_groups = (
    [id(p) for p in muon_params] + [id(p) for p in adamw_params]
)
assert len(all_param_ids_in_groups) == len(set(all_param_ids_in_groups)), \
    "有参数被分配到多个优化器组！"

# 找到 tok_embeddings 和 lm_head 的参数指针
tok_emb_id  = id(dict(model.named_parameters())['model.tok_embeddings.weight'])
lm_head_ids = [id(p) for n, p in model.named_parameters() if 'lm_head' in n]
print(f"\ntok_embeddings.weight id : {tok_emb_id}")
print(f"lm_head 相关 param ids    : {lm_head_ids}")
print(f"ids 相同（已 tied）        : {tok_emb_id in lm_head_ids}")
print(f"在 AdamW 组中出现次数      : {sum(id(p) == tok_emb_id for p in adamw_params)}"
      f"  （应为 1，避免重复梯度更新）")

### 5. 构造优化器并执行一步

In [ ]:
optimizers = build_walkie_optimizers(
    model,
    adamw_lr=3e-4,
    muon_lr=2e-2,
    adamw_weight_decay=0.1,
    muon_momentum=0.95,
)

# 模拟一次前向 + 反向
idx     = torch.randint(0, cfg.vocab_size, (2, 8))
targets = torch.randint(0, cfg.vocab_size, (2, 8))
_, loss = model(idx, targets)
loss.backward()

# 执行一步优化
for opt in optimizers.values():
    opt.step()
    opt.zero_grad()

print("loss:", round(float(loss), 4))
print("optimizers:", list(optimizers.keys()))
for name, opt in optimizers.items():
    print(f"  {name}:  {type(opt).__name__}  lr={opt.param_groups[0]['lr']:.2e}")

### 6. 源码精讲

**`zeropower_via_newton_schulz5`**：

```python
def zeropower_via_newton_schulz5(G, steps=5, eps=1e-7):
    a, b, c = 3.4445, -4.7750, 2.0315
    # 短边在前：若 m > n，对转置矩阵 G^T 做迭代，减小 X@X^T 的大小
    transposed = G.size(0) > G.size(1)
    X = G.t() if transposed else G
    # CUDA 上用 bfloat16 降低显存；CPU 上用 float32 保数值稳定
    X = X.to(torch.bfloat16 if G.is_cuda else torch.float32)
    X = X / (X.norm() + eps)    # 谱范数归一：确保 X₀ 的奇异值 ≤ 1，迭代收敛
    for _ in range(steps):
        A = X @ X.t()            # A = X Xᵀ，shape (min, min)
        B = b * A + c * (A @ A)  # 5 阶多项式的偶次项
        X = a * X + B @ X        # X_{k+1} = a X + (b A + c A²) X
    return (X.t() if transposed else X).to(G.dtype)
```

**`Muon.step()`** 的参数更新流程：

```python
# 1. Nesterov 动量
buf = buf * momentum + grad
update = grad + buf * momentum   # Nesterov：预测下一步后再用梯度修正

# 2. 极分解正交化
ortho = zeropower_via_newton_schulz5(update, steps=ns_steps)

# 3. 尺度因子：补偿宽高比，使高 fan_out/fan_in 的矩阵不会步长偏小
fan_out, fan_in = p.shape
scale = max(1.0, (fan_out / fan_in) ** 0.5)

# 4. 解耦权重衰减 + 参数更新
p.mul_(1.0 - lr * wd)            # 先 shrink（与梯度更新解耦）
p.add_(ortho, alpha=-lr * scale) # 再沿正交方向更新
```

**`split_walkie_params` 与 tied weights 处理**：

```python
seen: set[int] = set()
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if id(p) in seen:        # 同一 Parameter 对象已分配，跳过（处理 tied weights）
        continue
    seen.add(id(p))
    if _is_muon_param(name, p):
        muon.append(p)
    else:
        adamw.append(p)      # tok_embeddings + lm_head（tied）只进 AdamW 一次
```

**`_is_muon_param` 分组规则**：

```python
def _is_muon_param(name, param):
    if param.ndim != 2:            # 只作用于 2D 矩阵
        return False
    lname = name.lower()
    if "embed" in lname or "lm_head" in lname:
        return False               # embedding 行稀疏，lm_head 通常 tied，不用 Muon
    if "norm" in lname:
        return False               # RMSNorm scale 是 1D 向量
    return True                    # q_proj/k_proj/v_proj/o_proj + gate/up/down ✓
```

**`build_walkie_optimizers`**（工厂函数，返回 dict）：

```python
def build_walkie_optimizers(model, *, adamw_lr, muon_lr, ...):
    muon_params, adamw_params, _, _ = split_walkie_params(model)
    return {
        "adamw": torch.optim.AdamW(adamw_params, lr=adamw_lr, ...),
        "muon":  Muon(muon_params, lr=muon_lr, momentum=0.95, nesterov=True, ...),
    }
# 训练循环调用方式：
# for opt in optimizers.values(): opt.step(); opt.zero_grad()
```

---

## 延伸阅读与参考资料

### 论文与博客
- **Muon Optimizer**: Keller Jordan, 2024. [GitHub](https://github.com/KellerJordan/Muon)
- **极分解 Newton-Schulz 收敛性**: Higham, 1986. *Computing the polar decomposition—with applications.*
- **Shampoo（矩阵预条件器先驱）**: Gupta et al., 2018. [arXiv:1802.09568](https://arxiv.org/abs/1802.09568)
- **Decoupled Weight Decay (AdamW)**: Loshchilov & Hutter, 2019. [arXiv:1711.05101](https://arxiv.org/abs/1711.05101)

### 工程实现
- **PyTorch AdamW**: [docs](https://pytorch.org/docs/stable/generated/torch.optim.AdamW.html)